# Compound clustering

## Aim
Similar compounds might bind to the same targets and show similar effects. Based on this similar property principle, compound similarity can be used to build chemical groups via clustering. From such a clustering, a diverse set of compounds can also be selected from a larger set of screening compounds for further experimental testing.

## For more details
https://github.com/volkamerlab/teachopencadd/blob/master/teachopencadd/talktorials/T005_compound_clustering/talktorial.ipynb

## Instructions
Replace XXX with the appropriate code

## Configuration

In [ ]:
# Imports
# 1. Standard library imports
from pathlib import Path
import sys
sys.path.append('../my_modules') # to tell where to find local modules

# 2. Third-party library imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
from rdkit import Chem, DataStructs
from rdkit.Chem import (
    Draw,
    PandasTools,
    rdFingerprintGenerator,
)
from rdkit.ML.Cluster import Butina
PandasTools.RenderImagesInAllDataFrames(images=True) # to molecules as images in DataFrames
from rdkit.Chem.Draw import IPythonConsole # needed to show molecules
# from rdkit.Chem.Draw.MolDrawing import MolDrawing, DrawingOptions # only needed if modifying defaults

import seaborn as sns

# 3. Local application imports
import kernel_infos

In [ ]:
# Information about the kernel
kernel_infos.show_kernel_info()

In [ ]:
# Global variables
HERE = Path().resolve()
print(f'{HERE}')
ROOT = HERE.parent
print(f'{ROOT}')
DATA = ROOT / 'data'
print(f'{DATA}')

## Data

In [ ]:
# File to load
EGFR_compounds_lipinski_csv_path = DATA / "EGFR_compounds_lipinski.csv"

In [ ]:
# Load data in a dataframe called egfr_df
egfr_df = pd.XXX(EGFR_compounds_lipinski_csv_path, index_col=0)

In [ ]:
# Check
print(f"egfr_df shape: {egfr_df.shape}")
egfr_df.head(2)

In [ ]:
# Create a list coumpounds_l of tuples (RDKit molecule, molecule_chembl_id)

# Initialise compounds_l
compounds_l = XXX

# Iterate over the dataframe egfr_df with itertuples() - return a tuple (index, col1, col2, ...)
for _, chembl_id, smiles in egfr_df[['molecule_chembl_id', 'smiles']].XXX:
    compounds_l.append((Chem.MolFromSmiles(smiles), chembl_id))

# List the 5 first elements of compounds_l
compounds_l[XXX]

## Calculate fingerprints

In [ ]:
# Create RDKit fingerprints for all molecules
rdkit_gen = XXX.GetRDKitFPGenerator(maxPath=5)
rdkit_fps_l = [rdkit_gen.GetFingerprint(mol) for mol, idx in compounds_l]

# How many compounds/fingerprints do we have?
print("Number of compounds converted:", XXX(rdkit_fps_l))
# What is the lengh of a fingerprint ?
print("Fingerprint length per compound:", XXX(rdkit_fps_l[0]))

## Tanimoto similarity and distance matrix

In [ ]:
# Caculate Tanimoto similarity of first two fingerprints 
Tc_sim = DataStructs.XXX(rdkit_fps_l[0], rdkit_fps_l[1])

# Print Tanimoto similarity and distance
print(f"Tanimoto similarity: {XXX:.2f}\nDistance: {XXX:.2f}")

In [ ]:
# Define a function to calculate a tanimoto distance matrix, implemented as a list, from a list of fingerprints
def tanimoto_distance_matrix(fps_l):
     """Calculate distance matrix for fingerprint list"""
     # initialise a list
     dist_matrix = XXX
     # in a loop from 1 to len(fps_l) for i given
     for XXX in range(1, XXX):
          # calculate tanimoto similarity between fp_i and fps_list of index 1 to i excluded
          similarities = XXX.BulkTanimotoSimilarity(fps_l[i], fps_l[:i])
          # add distances (1 - Tanimoto) to the distance matrix
          dist_matrix.extend([XXX for Tc in similarities])
     # Return dist_matrix
     return dist_matrix

In [ ]:
# Calculate matrix
distance_matrix_egfr = XXX(rdkit_fps_l)

In [ ]:
# Print the first five values
for XXX in distance_matrix_egfr[0:5]:
    print(f"{elt:.5f}")

In [ ]:
# Calculate the number of element of the matrix
n_fps = XXX(rdkit_fps_l)
print(f"Number of fingerprints: {n_fps}")

# n elts in matrix
print(f"Number of distances, read: {XXX(distance_matrix_egfr)}")

# calculate the number of elements in the triangular matrix = sum (1, 2, 3, ..., n_fps-1) 
print(f"Number of distances, computed: {int(XXX)}")

## Clustering

In [ ]:
# Define a clustering function
def cluster_fps(fps_l, cutoff=0.2):
    # calculate Tanimoto distance matrix
    distance_matrix = XXX(fps_l)
    # cluster data with Butina algorithm
    clusters = XXX.ClusterData(
        distance_matrix, 
        len(fps_l),
        cutoff,
        isDistData=True)
    return clusters

In [ ]:
# Cluster egfr dataset with a distance threshold of 0.3
dist_threshold = XXX
clusters = cluster_fps(
    fps_l=rdkit_fps_l,
    cutoff=XXX)

In [ ]:
# What is the type of clusters ?
print(f'Type of clusters: {XXX(clusters)}')

# How many clusters ?
print(f"Number of clusters: '{XXX(clusters)}")

# What is the type of a cluster, ex with the first cluster ?
print(f'Type of a cluster: {type(XXX)}')

# Define c3_l, a list of all clusters with length of 3
c3_l = [clust for clust in clusters if len(clust) == XXX]

# Print size of c3_l
print(f"Size of c3_l: {XXX(c3_l)}")

# Print the first cluster of c3_l
print(XXX"First cluster: {XXX}")

In [ ]:
# Fix distance threshold to 1
dist_threshold = 1.0

# Cluster egfr dataset
clusters = XXX(rdkit_fps_l, cutoff=dist_threshold)

# How many clusters ?
print(f"Number of clusters : {len(XXX)}")

# How many compounds in the first cluster ?
print(f"Number of compounds in cluster #0 : {XXX}")

**What conclusions can be drawn for a dist_threshold of 1 ?**  
XXX

In [ ]:
# Fix distance threshold to 0
dist_threshold = 0.0

# Cluster egfr dataset
clusters = cluster_fps(rdkit_fps_l, cutoff=dist_threshold)

# How many clusters ?
print(f"Number of clusters : {len(clusters)}")

# How many compounds in ten randomly selected clusters ?
for c in random.sample(clusters,10):
    print(f"{len(c)}")

**What conclusions can be drawn for a dist_threshold of 0 ?**  
XXX

## How to pick a reasonable cutoff ?

In [ ]:
# Define a function to plot the size of clusters
def plot_cluster_sizes(clusters, cutoff):
    plt.figure(1, figsize=(10, 4))
    plt1 = plt.subplot(111)
    plt.axis([0, len(clusters), 0, len(clusters[0])+1])
    plt.xlabel('Cluster index', fontsize=16)
    plt.ylabel('Number of molecules', fontsize=16)
    plt.title(f'Distance threshold: {cutoff:.2f}')
    plt.tick_params(labelsize=16)
    plt1.bar(
        range(1, len(clusters)), 
        [len(c) for c in clusters[:len(clusters)-1]],
        lw=0
    )
    plt.show()

In [ ]:
# Plot cluster size distribution
plot_cluster_sizes(clusters, dist_threshold)

In [ ]:
# Plot size distribution varying threshold from 0.0 to 1.1 with a step of 0.1
deb = XXX
fin = XXX
step = XXX
for threshold in np.arange(start=deb, stop=fin, step=step):
    plot_cluster_sizes(cluster_fps(rdkit_fps_l, threshold), threshold)

## Cluster selection

In [ ]:
# Select cluster corresponding to a distance threshold of 0.3
dist_threshold = XXX
clusters = XXX(rdkit_fps_l, dist_threshold)

In [ ]:
# Number of clusters
print(f"Number of clusters: {XXX(clusters)}")

# Number of singletons (1 molecule in the cluster)
singletons_l = [c for c in clusters if XXX]
print(f"Number of singletons: {len(singletons_l)}")

# Number of molecules in the largest cluster
print(f"Number of molecules in the largest cluster: {len(clusters[XXX])}")

In [ ]:
# Tanimoto similarity between the first 2 compounds of cluster #O
print(f"Intra Tanimoto similarity: {DataStructs.TanimotoSimilarity(rdkit_fps_l[clusters[XXX][XXX]], rdkit_fps_l[clusters[XXX][XXX]]):.2f}")

# Tanimoto similarity between the centroids of cluster #O and cluster #1
print(f"Inter Tanimoto similarity: {DataStructs.TanimotoSimilarity(rdkit_fps_l[clusters[XXX][XXX]], rdkit_fps_l[clusters[XXX][XXX]]):.2f}")

## Cluster visualisation

In [ ]:
# Draw an image of the first 10 molecules from cluster #0, add their ChEMBL_ID on legend and the number of the cluster
n_cluster = 0
nb_mols = 10
Draw.MolsToGridImage(
    [compounds_l[ind][0] for ind in clusters[n_cluster][:nb_mols]],
    XXX = [f"#{n_cluster} {compounds_l[ind][1]}" for ind in clusters[n_cluster][:nb_mols]],
    molsPerRow=5
)

In [ ]:
# Save molecules from largest cluster (#0) in a sdf file
cluster0_sdf_path = str(DATA / "cluster0.sdf")
sdf = Chem.SDWriter(cluster0_sdf_path)
for index in clusters[0]:
    mol, label = compounds_l[index]
    # Set label as the property "_Name"
    mol.XXX("_Name", label)
    # Write mol to the sdf file
    sdf.XXX(mol)
# Close the sdf file
sdf.XXX

In [ ]:
# Draw an image of the first 10 molecules from cluster #1, add their ChEMBL_ID on legend and the number of the cluster
n_cluster = XXX
nb_mols = XXX
Draw.XXX(
    [compounds_l[ind][0] for ind in clusters[n_cluster][:nb_mols]],
    legends = [f"#{n_cluster} {compounds_l[ind][1]}" for ind in clusters[n_cluster][:nb_mols]],
    molsPerRow=5
)

In [ ]:
# Draw an image of the first 5 centroids, add their ChEMBL_ID on legend and the number of the cluster
nb_clusters = 5
Draw.MolsToGridImage(
    [compounds_l[clusters[ind][0]][0] for ind in range(0, nb_clusters)],
    legends = [f"#{ind} {compounds_l[clusters[ind][0]][1]}" for ind in range(0, nb_clusters)],
    molsPerRow=3
)

## Similarity within the first 10 clusters (intra-cluster similarity)

### Calculation

In [ ]:
# Define a function to calculate Tanimoto similarity for all pairs of fingerprints in each cluster
def intra_tanimoto_similarity(fps_clusters):
    intra_similarity = list()
    # Calculate intra similarity per cluster
    for cluster in fps_clusters:
        intra_similarity.append([1 - x for x in tanimoto_distance_matrix(cluster)]) # convert distance similarity in Tanimoto similarity and append a list for each cluster
    return intra_similarity # list of list of Tanimoto similarity per cluster



In [ ]:
# Recompute fingerprints for the first 10 selected clusters
fps_per_cluster = list()
for cluster in clusters[:10]:
    fps_per_cluster.append([rdkit_gen.XXX(compounds_l[ind][0]) for ind in cluster])

In [ ]:
# Check fps_per_cluster
print(f"Type of fps_per_cluster: {XXX(fps_per_cluster)}")
print(f"Number of elements in fps_per_cluster : {XXX(fps_per_cluster)}")
print(f"Type of first element of fps_per_cluster: {type(XXX)}")

In [ ]:
# Calculate intra-cluster tanimoto similarity
intra_tanimoto_sim = XXX(fps_per_cluster)

In [ ]:
# Check intra_tanimoto_sim
print(f"Type of intra_tanimoto_sim: {type(intra_tanimoto_sim)}")
print(f"Number of elements in intra_tanimoto_sim : {len(intra_tanimoto_sim)}")
print(f"Type of first element of intra_tanimoto_sim: {type(intra_tanimoto_sim[0])}")

### Visualisation

In [ ]:
# Create a dataframe sim_df
# x column : similarity values
# y column : index of the corresponding cluster
sim_l = [] # list of dictionnaries {x,y}
for ind, cluster_sim in enumerate(intra_tanimoto_sim, start=1): # to associate the cluster index, starting from 1, to the similarity sub-list 
    for sim in cluster_sim:
        sim_l.append({"x":ind, "y":sim}) # fulfill the list of dictionnaries

sim_df = pd.XXX(sim_l)

In [ ]:
# Check
print(f"sim_df.shape: {sim_df.shape}")
sim_df.head(2)

In [ ]:
# Violin plot with seaborn (sns)
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 6))

ax = XXX.violinplot(
    x="x", 
    y="y", 
    data=sim_df,
    inner="quartile",  # Show quartiles and median
    hue="x",           # Use 'hue' to avoid the warning
    palette="pastel",  # Color palette for violins
    legend=False,      # Disable redundant legend
    linewidth=1.5,     # Line thickness
)

# Customize the color of the lines (median and quartiles)
for line in ax.lines:
    line.set_color("red")      # Color of the lines (median and quartiles)
    line.set_linewidth(2)      # Line thickness

# Customize the border of the violins
for violin in ax.collections:
    violin.set_edgecolor("purple")  # Border color

# Manually add the mean
means = sim_df.groupby("x")["y"].mean()
for x, mean in means.items():
    plt.scatter(x=x-1, y=mean, color="green", s=100, label=f"Mean (x={x})")

# Set custom x-axis and y-axis labels
plt.xlabel("Cluster index")  # Replace with your desired x-axis label
plt.ylabel("Similariy")  # Replace with your desired y-axis label

plt.title("Intra-cluster Tanimoto similarity (Median/Quartiles in Red and Mean in Green)")
plt.show()

## Get the centroids compounds

In [ ]:
# Get a centroid list (cluster centers: first compound in each cluster)
centroids_l = [compounds_l[XXX] for c in clusters]

In [ ]:
# Check
print(f"Number of centroids: {len(centroids_l)}")
print(f"Number of clusters: {XXX(clusters)}")

Remark

[A more complex selection is described in TOC.](https://github.com/volkamerlab/teachopencadd/blob/master/teachopencadd/talktorials/T005_compound_clustering/talktorial.ipynb)